# EasyColab · Penny Royal 浏览器控制台
保存到你自己的 Google Drive，并收藏这份笔记本。换电脑只需登录同一个 Google 账号。

## 每次使用
1. **文件 → 在云端硬盘中保存副本**（第一次）。在 **代码执行程序 / Runtime → 更改运行时类型** 选择 **G4 / RTX PRO 6000 96GB**，点击右上角 **连接**。分配 GPU 由 Colab 完成，受额度和机型库存限制；本笔记本不能在没有运行时的情况下替你分配 GPU。
2. 左侧 **钥匙 / Secrets** 添加 `CLOUDFLARE_TUNNEL_TOKEN` 和 `EASYCOLAB_API_KEY`，打开本笔记本访问权限。API Key 可以复用 EasyColab App 中的密钥，至少 16 个字符。**不要将密钥写入代码或分享笔记本输出。**
3. 按顺序运行下面三个代码单元格。在控制面板填固定域名，然后点击 **① 准备环境 → ② 启动 API → ③ 加载模型**。首次编译和云盘到 SSD 复制会较久；GPU 已分配就会消耗额度。
4. 在 Open WebUI / Magpie 使用 `https://你的域名/v1`，API Key 为上述 Secret，模型 ID 为 `jpezzulli/OrcaRouter-Qwen3.8-Flash-Next-Uncensored-ModelOpt-NVFP4`。此控制台默认 4K，可选 8K/16K/32K。
5. 用完勾选 **确认释放当前笔记本 GPU**，点击红色 **释放 GPU 实例**。仅关闭网页或卸载模型不会停止消耗额度。

## 安全及复用范围
- 只支持已有的这套 ModelOpt 权重，不会误下载旧 vLLM 模型；默认读取真实 Drive 的 `MyDrive/EasyColab_Drive/models`，这是桌面 App 以前保存的位置。若文件在 `MyDrive/models`，修改面板路径。权重缺失会停止，不会静默重新下载 126 GiB。
- Drive 官方授权由 Colab 弹窗处理，不需要 rclone。权重只读，从 Drive 复制到 SSD 再加载；不会删除或覆盖云盘模型。大小及固定版本清单校验不是全文件内容哈希。
- 固定使用 EasyColab 源码提交 `94b32ca`（含 SSE 断连修复）及 Penny Royal 3.1.6。不会随着远程分支变化自动执行新代码。系统工具链和编译缓存仍是临时的，释放实例后需要重新准备。
- Cloudflare Published application 指向 `http://localhost:7860`，Path 留空。**先停止同一 Tunnel 上的旧 EasyColab 实例**，避免同一域名被分发到两台不同模型服务器。
- 本模式是独立的、带鉴权的 API 服务，不是旧 MiniMax 网页，也不提供桌面 App 的完整管理接口。域名根地址不是启动控制台；停机后必须回到本笔记本启动。所有 API 和控制路由都需要密钥，建议继续限制可信来源。
- 默认闲置 30 分钟自动释放；准备环境期间尚未启动此看门狗，不使用时请手动释放。不要同时用两份笔记本控制同一实例。**运行全部只显示面板，不自动开始编译、加载或释放。**


In [ ]:
import os
from google.colab import drive

if os.path.ismount('/content/drive/MyDrive'):
    raise RuntimeError('当前是桌面端 rclone 挂载。请先结束旧实例，再为本笔记本分配独立 GPU；不会重挂或覆盖已有目录。')
drive.mount('/content/drive')
if not os.path.ismount('/content/drive') or not os.path.isdir('/content/drive/MyDrive'):
    raise RuntimeError('Google Drive 未挂载成功，请完成授权后重试。')
os.chdir('/content')
print('Drive 已挂载。下一步运行控制程序单元格和面板单元格。')


In [ ]:
%%writefile /content/easycolab_control.py
import hashlib
import hmac
import json
import os
from pathlib import Path
import re
import shutil
import signal
import socket
import subprocess
import sys
import tempfile
import threading
import time
import urllib.error
import urllib.request

SOURCE_REVISION = '94b32caae01a13f249024e65b32688f20de6b18e'
MODEL_REPO = 'jpezzulli/OrcaRouter-Qwen3.8-Flash-Next-Uncensored-ModelOpt-NVFP4'
MODEL_REVISION = 'f24d2b68ff2814f24455ae86717be276619b5664'
MODEL_ID = 'qwen38_flash_next_modelopt_sglang'
ROOT = Path('/content/easycolab-notebook-source')
PYTHON = Path('/content/.easycolab-control-env/bin/python')
LOG = Path('/content/easycolab_control.log')
SOURCES = {
    'app.py': 'bd151ca61ce36ef029adbf39c1c073b882b24da9dff242f3aef70a9db58ee5b3',
    'easycolab_sglang.py': '349b7a8bce9d2432587939b1ab0af372d2fac5b2e6cbfaaca72a577545234240',
    'easycolab_vllm.py': '585f02c0a993f5d5e543694875667e11ee5efa7833b90d914b84f37259269560',
    'easycolab_tunnel.py': '15d98424cdfb97e180055ccdd54b574f11d2abbaefc78a3027f537008e8e2fee',
    'easycolab_monitor.py': 'e9cbe30e6c2386826b06b9395823dcd4897f0cef51805c3c0d1828d36d6736e5',
    'easycolab_transfer.py': '077063e6c991c91329f0106bfaa5d400a0b555b4acb2de87ab0510c0bccbaf72',
}

def validate_config(config):
    if not re.fullmatch(r'https://(?:[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?\.)+[A-Za-z]{2,63}', config['public_url']):
        raise ValueError('固定域名须为 https://ai.example.com，不含路径、端口或参数。')
    for name in ('api_key', 'tunnel_token'):
        value = config.get(name, '')
        if not value or any(character.isspace() or ord(character) < 32 for character in value) or len(value) > 4096:
            raise ValueError('请检查 Colab Secrets 中的 API Key 和 Tunnel Token 格式。')
    if len(config['api_key']) < 16:
        raise ValueError('EASYCOLAB_API_KEY 至少需要 16 个字符。')
    drive_root = Path('/content/drive/MyDrive').resolve()
    model_root = Path(config['model_root']).resolve()
    if model_root == drive_root or not model_root.is_relative_to(drive_root):
        raise ValueError('模型目录必须是 /content/drive/MyDrive 下的子目录。')
    if config['context'] not in (4096, 8192, 16384, 32768) or not 5 <= config['idle_minutes'] <= 120:
        raise ValueError('请选择 4K–32K 上下文和 5–120 分钟闲置释放时间。')

def check_mount():
    if not os.path.ismount('/content/drive') or not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('请先运行官方 Drive 挂载单元格，不会回退临时模型目录。')

def fetch_sources():
    ROOT.mkdir(parents=True, exist_ok=True)
    for name, expected in SOURCES.items():
        path = ROOT / name
        if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected:
            continue
        url = f'https://raw.githubusercontent.com/buaazp/easycolab/{SOURCE_REVISION}/{name}'
        with urllib.request.urlopen(url, timeout=45) as response:
            payload = response.read(2 * 1024 * 1024)
        if hashlib.sha256(payload).hexdigest() != expected:
            raise RuntimeError('固定版本源码校验失败，未执行下载的代码。')
        temporary = path.with_suffix('.download')
        temporary.write_bytes(payload)
        os.replace(temporary, path)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

def weight_sizes():
    from easycolab_sglang import MODEL_FILES
    return dict(MODEL_FILES)

def safe_path(root, name):
    path = root / name
    if not isinstance(name, str) or not name or Path(name).is_absolute() or '..' in Path(name).parts or path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):
        raise ValueError('模型清单含不安全路径，未读取或复制。')
    return path

def check_weights(source):
    check_mount()
    source = Path(source)
    manifest_path = source / '.easycolab-metadata.json'
    if not manifest_path.is_file() or manifest_path.is_symlink():
        raise RuntimeError('找不到已保存的模型清单。请确认 EasyColab_Drive/models 与 models 的区别；不会自动重新下载大模型。')
    manifest = json.loads(manifest_path.read_text())
    if manifest.get('repo_id') != MODEL_REPO or manifest.get('revision') != MODEL_REVISION:
        raise ValueError('云盘权重不是指定版本的 jpezzulli ModelOpt 模型。')
    metadata = manifest.get('files')
    if not isinstance(metadata, dict) or not metadata or 'config.json' not in metadata:
        raise ValueError('配置文件清单不完整，请在桌面端校验权重。')
    sizes = {**metadata, **weight_sizes()}
    for name, size in sizes.items():
        path = safe_path(source, name)
        if not isinstance(size, int) or size <= 0 or not path.is_file() or path.stat().st_size != size:
            raise ValueError('权重或配置缺失/大小不符: ' + name)
    return manifest, sizes

def stage_weights(source, destination, report=print):
    source, destination = Path(source), Path(destination)
    manifest, sizes = check_weights(source)
    if destination.is_symlink() or destination.resolve().is_relative_to(Path('/content/drive').resolve()):
        raise ValueError('SSD 缓存不能指向云盘或软链接。')
    destination.mkdir(parents=True, exist_ok=True)
    required = 0
    for name, size in sizes.items():
        target = safe_path(destination, name)
        if not target.is_file() or target.stat().st_size != size:
            required += size
    if shutil.disk_usage(destination).free < required + 22 * 1024**3:
        raise RuntimeError(f'SSD 还需 {required / 1024**3:.1f} GiB 加 22 GiB 余量；不会自动删除旧 vLLM 权重。')
    weights = weight_sizes()
    for index, (name, size) in enumerate(sizes.items(), 1):
        check_mount()
        target = safe_path(destination, name)
        if name in weights and target.is_file() and target.stat().st_size == size:
            continue
        report(f'SSD 预热 {index}/{len(sizes)}: {name}（云盘只读）')
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = safe_path(destination, name + '.staging')
        with safe_path(source, name).open('rb') as reader, temporary.open('wb') as writer:
            shutil.copyfileobj(reader, writer, 8 * 1024 * 1024)
        if temporary.stat().st_size != size:
            raise RuntimeError('SSD 复制大小不符，可重试: ' + name)
        os.replace(temporary, target)
    manifest_target = safe_path(destination, '.easycolab-metadata.json')
    manifest_target.write_text(json.dumps(manifest))
    return str(destination)

def occupied():
    with socket.socket() as probe:
        probe.settimeout(1)
        return probe.connect_ex(('127.0.0.1', 7860)) == 0

def clean_environment():
    environment = os.environ.copy()
    for name in ('HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'EASYCOLAB_API_KEY', 'CLOUDFLARE_TUNNEL_TOKEN', 'TUNNEL_TOKEN', 'PYTHONPATH', 'PYTHONHOME'):
        environment.pop(name, None)
    environment.update(DEBIAN_FRONTEND='noninteractive', EASYCOLAB_PUBLIC_URL='', EASYCOLAB_SGLANG_CUDA_HOME='/usr/local/cuda-13.3', EASYCOLAB_SGLANG_CC='/usr/bin/gcc-14', EASYCOLAB_SGLANG_CXX='/usr/bin/g++-14')
    environment['PATH'] = str(Path.home() / '.cargo/bin') + os.pathsep + environment.get('PATH', '')
    return environment

def run_command(command, label, timeout=900):
    print(label, flush=True)
    with open(LOG, 'ab') as output:
        os.chmod(LOG, 0o600)
        process = subprocess.Popen(command, cwd='/content', env=clean_environment(), stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            if process.wait(timeout=timeout):
                raise RuntimeError(label + '失败，点击查看日志；GPU 尚未释放。')
        except subprocess.TimeoutExpired:
            raise RuntimeError(label + '超时，点击查看日志；GPU 尚未释放。') from None
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait(timeout=5)

def prepare(config):
    validate_config(config)
    check_mount()
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError('此固定配方需要 Colab Python 3.12，请选择匹配的运行时版本。')
    if occupied():
        raise RuntimeError('7860 已有服务。不要覆盖运行中的环境；如是本控制台，直接刷新状态或加载。')
    for name, value in clean_environment().items():
        if name.startswith('EASYCOLAB_SGLANG_'):
            os.environ[name] = value
    fetch_sources()
    import easycolab_sglang as backend
    import torch
    backend.validate_hardware(torch)
    backend.validate_memory(available_ram())
    check_weights(Path(config['model_root']) / MODEL_REPO)
    if shutil.disk_usage('/content').free < 25 * 1024**3:
        raise RuntimeError('环境安装至少预留 25 GiB SSD，模型预热还需约 126 GiB。')
    if not backend.preflight()['ok']:
        system = dict(line.strip().split('=', 1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
        if os.geteuid() != 0 or system.get('ID', '').strip(chr(34)) != 'ubuntu' or system.get('VERSION_ID', '').strip(chr(34)) != '24.04':
            raise RuntimeError('自动工具链安装仅支持 Colab Ubuntu 24.04 root，请勿在个人电脑运行。')
        run_command(['apt-get', 'update', '-qq'], '更新系统包索引')
        run_command(['apt-get', 'install', '-y', '--no-install-recommends', 'cuda-toolkit-13-3', 'gcc-14', 'g++-14', 'build-essential', 'pkg-config', 'libssl-dev', 'protobuf-compiler', 'python3.12-dev', 'python3.12-venv', 'ninja-build', 'git', 'curl'], '安装 CUDA 13.3 / GCC14（不安装或升级驱动）', 1800)
        if not shutil.which('cargo', path=clean_environment()['PATH']):
            script = '/content/easycolab-rustup.sh'
            run_command(['curl', '--fail', '--location', '--proto', '=https', '--tlsv1.2', 'https://sh.rustup.rs', '-o', script], '获取 Rust 官方安装器')
            run_command(['sh', script, '-y', '--profile', 'minimal', '--default-toolchain', 'stable'], '安装 Rust/Cargo')
    if not PYTHON.is_file():
        run_command([sys.executable, '-m', 'venv', '--system-site-packages', str(PYTHON.parent.parent)], '创建隔离的 API 环境')
    run_command([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', 'gradio', 'fastapi', 'uvicorn', 'pillow', 'requests', 'av', 'hf_transfer', 'librosa', 'soundfile'], '安装 API 环境依赖')
    backend.ensure_runtime(print)
    print('环境已就绪。下一步点击启动 API，再点击加载模型。', flush=True)

def available_ram():
    for line in Path('/proc/meminfo').read_text().splitlines():
        if line.startswith('MemAvailable:'):
            return int(line.split()[1]) / 1024**2
    return 0

def request(config, path, body=None, timeout=20):
    payload = json.dumps(body).encode() if body is not None else None
    message = urllib.request.Request('http://127.0.0.1:7860' + path, data=payload, headers={'Authorization': 'Bearer ' + config['api_key'], 'Content-Type': 'application/json'})
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        with opener.open(message, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'控制接口 HTTP {error.code}；检查 Secret 是否与当前实例一致，或等待进行中的操作结束。') from None

def show_logs(config):
    if not LOG.exists():
        print('暂无控制台日志。')
        return
    from easycolab_tunnel import redact_diagnostic
    with LOG.open(errors='replace') as stream:
        stream.seek(max(0, LOG.stat().st_size - 9000))
        text = stream.read()
    print(redact_diagnostic(text, (config['api_key'], config['tunnel_token'])))

def start(config):
    validate_config(config)
    check_mount()
    if occupied():
        status = request(config, '/control/status')
        if status.get('source_revision') != SOURCE_REVISION or status.get('model_root') != config['model_root'] or status.get('context') != config['context'] or status.get('public_url') != config['public_url']:
            raise RuntimeError('已有服务与面板配置不同，不会接管或重启；请检查原实例。')
        print('复用本控制台已运行的 API。', flush=True)
        return status
    if not PYTHON.is_file():
        raise RuntimeError('请先准备环境。')
    fetch_sources()
    descriptor, path = tempfile.mkstemp(prefix='easycolab-control-', suffix='.json', dir='/content')
    process = None
    try:
        with os.fdopen(descriptor, 'w') as stream:
            json.dump(config, stream)
        os.chmod(path, 0o600)
        with LOG.open('ab') as log:
            os.chmod(LOG, 0o600)
            process = subprocess.Popen([str(PYTHON), '-u', '/content/easycolab_control.py', 'serve', path], cwd='/content', env=clean_environment(), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        deadline = time.monotonic() + 180
        while time.monotonic() < deadline and process.poll() is None:
            try:
                status = request(config, '/control/status', timeout=2)
                if status.get('tunnel_ready') and status.get('source_revision') == SOURCE_REVISION:
                    print('API 已启动: ' + config['public_url'] + '/v1；尚需点击加载模型。', flush=True)
                    return status
            except (OSError, RuntimeError):
                pass
            time.sleep(2)
        raise RuntimeError('API/Tunnel 启动失败或超时，查看日志；不会回退随机地址。')
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=12)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
        raise
    finally:
        Path(path).unlink(missing_ok=True)

def make_server(runtime, backend, config):
    from fastapi import FastAPI
    from fastapi.responses import JSONResponse
    runtime.API_AUTH_KEY = config['api_key']
    runtime.IDLE_TIMEOUT_MINUTES = config['idle_minutes']
    runtime.SELECTED_MODEL_ID = MODEL_ID
    runtime.QWEN_CONTEXT_LENGTH = config['context']
    runtime.touch_activity()
    api = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
    runtime.attach_macos_api_routes(api)
    allowed = {'/v1/models', '/v1/chat/completions', '/v1/messages', '/v1/messages/count_tokens'}
    api.router.routes[:] = [route for route in api.router.routes if route.path in allowed]
    status = {'source_revision': SOURCE_REVISION, 'public_url': config['public_url'], 'model_root': config['model_root'], 'context': config['context'], 'phase': 'idle', 'message': 'API 就绪，等待加载模型', 'tunnel_ready': False}
    operation_lock = threading.Lock()

    class RequireKey:
        def __init__(self, app):
            self.app = app

        async def __call__(self, scope, receive, send):
            if scope['type'] == 'http':
                headers = dict(scope.get('headers', []))
                authorization = headers.get(b'authorization', b'').decode('latin1')
                provided = authorization[7:].strip() if authorization.lower().startswith('bearer ') else headers.get(b'x-api-key', b'').decode('latin1')
                if not hmac.compare_digest(provided.encode(), config['api_key'].encode()):
                    await JSONResponse(status_code=401, content={'error': {'message': 'Invalid API Key'}})(scope, receive, send)
                    return
            await self.app(scope, receive, send)

    api.add_middleware(RequireKey)

    def report(message):
        status['message'] = message
        runtime.touch_activity()
        print(message, flush=True)

    @api.get('/control/status')
    def state():
        bundle = runtime.LOADED_RUNTIMES.get(MODEL_ID, {}).get('llm_bundle')
        return {**status, 'loaded': runtime._is_bundle_alive(bundle), 'gpu_busy': runtime.GPU_LOCK.locked(), 'idle_minutes': runtime.IDLE_TIMEOUT_MINUTES}

    def load_worker():
        try:
            with runtime.GPU_LOCK:
                backend.validate_hardware(runtime.torch)
                backend.validate_memory(available_ram())
                free, total = runtime.torch.cuda.mem_get_info(0)
                if free < total * 0.9:
                    raise RuntimeError('可用显存不足 90%，请先结束其他 GPU 任务。')
                local = stage_weights(Path(config['model_root']) / MODEL_REPO, Path('/content/models') / MODEL_REPO, report)
                bundle = backend.start_server(local, config['context'], report)
                spec = backend.MODEL_SPEC
                runtime.LOADED_RUNTIMES[MODEL_ID] = {'model_id': MODEL_ID, 'name': spec['name'], 'modality': 'llm_vlm', 'mode_label': bundle['mode_label'], 'pipeline': None, 'llm_bundle': bundle, 'vram_gb': 0.0, 'loaded_at': time.time()}
                runtime._sync_legacy_globals()
                status['phase'] = 'ready'
                report('模型已加载。可在外部客户端使用完整 jpezzulli 模型 ID。')
        except Exception as error:
            from easycolab_tunnel import redact_diagnostic
            status['phase'] = 'failed'
            report(redact_diagnostic(str(error), (config['api_key'], config['tunnel_token']))[-2500:])
        finally:
            runtime.MODEL_DOWNLOAD_STATE['is_loading_vram'] = False
            runtime.touch_activity()
            operation_lock.release()

    @api.post('/control/load')
    def load():
        if not operation_lock.acquire(blocking=False):
            return JSONResponse(status_code=409, content={'error': '操作进行中'})
        if state()['loaded']:
            operation_lock.release()
            return state()
        status['phase'] = 'loading'
        runtime.MODEL_DOWNLOAD_STATE['is_loading_vram'] = True
        threading.Thread(target=load_worker, daemon=True).start()
        return state()

    @api.post('/control/unload')
    def unload():
        if not operation_lock.acquire(blocking=False):
            return JSONResponse(status_code=409, content={'error': '加载进行中；如需立即停止请释放当前实例'})
        try:
            runtime.unload_model_from_vram_core(MODEL_ID)
            status.update(phase='idle', message='模型已卸载；GPU 实例仍在消耗额度')
            return state()
        finally:
            operation_lock.release()

    return api, status

def serve(config_path):
    import fcntl
    import uvicorn
    config_file = Path(config_path)
    try:
        config = json.loads(config_file.read_text())
    finally:
        config_file.unlink(missing_ok=True)
    validate_config(config)
    check_mount()
    ownership = open('/content/.easycolab-control.lock', 'w')
    fcntl.flock(ownership, fcntl.LOCK_EX | fcntl.LOCK_NB)
    if occupied():
        raise RuntimeError('7860 端口已有服务，未覆盖。')
    sys.path.insert(0, str(ROOT))
    os.environ['EASYCOLAB_REQUIRE_DRIVE'] = '1'
    os.environ['EASYCOLAB_MODEL_DIR'] = config['model_root']
    os.environ['EASYCOLAB_PUBLIC_URL'] = ''
    os.environ.pop('CLOUDFLARE_TUNNEL_TOKEN', None)
    import app as runtime
    import easycolab_sglang as backend
    from easycolab_tunnel import CloudflareTunnel, exit_on_sigterm
    signal.signal(signal.SIGTERM, exit_on_sigterm)
    api, status = make_server(runtime, backend, config)
    server = uvicorn.Server(uvicorn.Config(api, host='127.0.0.1', port=7860, log_level='warning'))
    worker = threading.Thread(target=server.run, daemon=True)
    tunnel = CloudflareTunnel()
    try:
        worker.start()
        deadline = time.monotonic() + 30
        while not server.started and worker.is_alive() and time.monotonic() < deadline:
            time.sleep(0.2)
        if not server.started:
            raise RuntimeError('本地 API 启动失败。')
        tunnel.start(runtime._ensure_cloudflared_binary(), config['tunnel_token'])
        status['tunnel_ready'] = True
        print('CONTROL_READY ' + config['public_url'], flush=True)
        while worker.is_alive():
            if tunnel.process.poll() is not None:
                raise RuntimeError('Tunnel 已退出，请重新启动 API。')
            time.sleep(1)
    finally:
        status['tunnel_ready'] = False
        tunnel.stop()
        server.should_exit = True
        backend.cleanup()
        ownership.close()

if __name__ == '__main__':
    if len(sys.argv) != 3 or sys.argv[1] != 'serve':
        raise SystemExit('Use the Colab notebook controls.')
    serve(sys.argv[2])


## 控制面板
运行本单元格只创建按钮。修改配置后重新运行本单元格；不要在已运行的实例上切换域名或 Secret。加载期间用“查看状态”和“查看日志”检查进度，显示 `loaded: true` 后才能对话。**卸载模型仍计费；释放实例才停止占用 GPU。**


In [ ]:
import importlib.util
import json
import os
import threading
from google.colab import userdata, runtime
import ipywidgets as widgets
from IPython.display import display

os.chdir('/content')
spec = importlib.util.spec_from_file_location('easycolab_control', '/content/easycolab_control.py')
control = importlib.util.module_from_spec(spec)
spec.loader.exec_module(control)
domain = widgets.Text(value='', description='固定域名', placeholder='https://colab.x1x.bid', layout=widgets.Layout(width='95%'))
model_root = widgets.Text(value='/content/drive/MyDrive/EasyColab_Drive/models', description='模型目录', layout=widgets.Layout(width='95%'))
context = widgets.Dropdown(options=[('4K（推荐先验证）', 4096), ('8K', 8192), ('16K', 16384), ('32K', 32768)], description='上下文')
idle = widgets.BoundedIntText(value=30, min=5, max=120, description='闲置分钟')
output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', max_height='400px', overflow='auto'))
prepare_button = widgets.Button(description='① 准备环境', button_style='info')
start_button = widgets.Button(description='② 启动 API', button_style='info')
load_button = widgets.Button(description='③ 加载模型', button_style='success')
status_button = widgets.Button(description='查看状态')
logs_button = widgets.Button(description='查看日志')
test_button = widgets.Button(description='测试 API')
unload_button = widgets.Button(description='卸载模型（仍计费）', layout=widgets.Layout(width='200px'))
confirm_release = widgets.Checkbox(value=False, description='确认释放当前笔记本 GPU，临时缓存会丢失', indent=False, layout=widgets.Layout(width='95%'))
release_button = widgets.Button(description='释放 GPU 实例', button_style='danger')
action_lock = threading.Lock()
configuration = {}

def read_configuration():
    try:
        values = {'public_url': domain.value.strip().rstrip('/'), 'model_root': model_root.value.strip(), 'context': context.value, 'idle_minutes': idle.value, 'api_key': userdata.get('EASYCOLAB_API_KEY'), 'tunnel_token': userdata.get('CLOUDFLARE_TUNNEL_TOKEN')}
    except Exception:
        raise RuntimeError('请在 Colab 左侧 Secrets 添加两个密钥并授权本笔记本。不要把密钥粘贴到代码中。') from None
    control.validate_config(values)
    return values

def invoke(action):
    def clicked(button):
        if not action_lock.acquire(blocking=False):
            output.append_stdout('准备或启动操作仍在执行，可查看状态/日志，或确认后释放实例。\n')
            return
        try:
            values = read_configuration()
            configuration.clear()
            configuration.update(values)
        except Exception as error:
            output.append_stdout(str(error) + '\n')
            action_lock.release()
            return
        def work():
            try:
                with output:
                    result = action(values)
                    if result is not None:
                        print(json.dumps(result, ensure_ascii=False, indent=2))
            except Exception as error:
                output.append_stdout(type(error).__name__ + ': ' + str(error)[:1200] + '\n')
            finally:
                action_lock.release()
        threading.Thread(target=work, daemon=True).start()
    return clicked

def inspect(action):
    def clicked(button):
        try:
            values = configuration or read_configuration()
            with output:
                result = action(values)
                if result is not None:
                    print(json.dumps(result, ensure_ascii=False, indent=2))
        except Exception as error:
            output.append_stdout(type(error).__name__ + ': ' + str(error)[:1200] + '\n')
    return clicked

def release_clicked(button):
    if not confirm_release.value:
        output.append_stdout('请先勾选确认。此操作只释放当前笔记本实例，不删除 Drive 权重。\n')
        return
    output.append_stdout('正在请求 Colab 释放当前 GPU，连接随后会断开；请在管理会话中确认实例已消失。\n')
    runtime.unassign()

prepare_button.on_click(invoke(control.prepare))
start_button.on_click(invoke(control.start))
load_button.on_click(invoke(lambda values: control.request(values, '/control/load', {})))
status_button.on_click(inspect(lambda values: control.request(values, '/control/status')))
logs_button.on_click(inspect(control.show_logs))
test_button.on_click(invoke(lambda values: control.request(values, '/v1/chat/completions', {'model': control.MODEL_REPO, 'messages': [{'role': 'user', 'content': 'Reply with exactly OK.'}], 'max_tokens': 32, 'temperature': 0, 'chat_template_kwargs': {'enable_thinking': False}}, timeout=120)))
unload_button.on_click(invoke(lambda values: control.request(values, '/control/unload', {}, timeout=60)))
release_button.on_click(release_clicked)
display(widgets.VBox([domain, model_root, widgets.HBox([context, idle]), widgets.HBox([prepare_button, start_button, load_button]), widgets.HBox([status_button, logs_button, test_button, unload_button]), confirm_release, release_button, output]))


## 常见问题
- **网页断开 / 面板消失**：重新连接到同一笔记本，再运行控制程序和面板单元格，填入相同配置，点击“查看状态”或“启动 API”；已有服务不会被重启。如果实例已被销毁，需要重新运行 Drive 挂载及准备环境。
- **126 GiB 权重找不到**：先用 Colab 左侧文件浏览器确认路径；桌面版默认在 `drive/MyDrive/EasyColab_Drive/models/jpezzulli/...`，不是 `drive/MyDrive/models/jpezzulli/...`。不要重新下载或删除旧模型来碰运气。
- **环境安装失败**：查看日志，检查是否为 Ubuntu 24.04、GPU 是否是 SM120 96GB、NVIDIA CUDA 软件源是否可用。准备操作可重试，但不会升级宿主 GPU 驱动或覆盖旧 vLLM 环境。
- **加载很久**：第一次需要读取约 126 GiB 云盘数据到 SSD，再读取权重、编译内核、预热。可点击查看日志。SSD 缓存完整时跳过大权重复制。
- **API 401**：使用 `EASYCOLAB_API_KEY` 的值，不是 Tunnel Token；如果运行中修改了 Secret，旧进程仍持有原值。
- **同一域名时好时坏**：确认另一个 EasyColab 实例没有还连着同一个 Cloudflare Tunnel。
- **面板无法释放**：在 Colab 菜单 **代码执行程序 → 管理会话** 终止对应实例，或 **断开连接并删除运行时**。不要仅关闭浏览器。此操作丢弃 SSD 缓存，不删除 Drive 权重。
- **API 断流**：Magpie 的 EasyColab Provider 可启用“仅 HTTP/1.1”；此笔记本固定的网关代码已经含 5 秒空闲 SSE 心跳。不能靠心跳修复真实网络断开。
